Hardware:
- MSI Katana
- Intel Core i7-10750H
- NVIDIA GeForce RTX 3050 Laptop GPU

Firstly, we import all code dependencies that will be helpful later on the training process

In [1]:
import pandas as pd
import numpy as np
import os

# Plain Text Generation

In this section we generate a plain text containing all Python scripts included in the datalake. This will be useful in the next section, in order to properly train our gpt-2 fine tuned model

In [2]:
def read_file(path):
  try:
      with open(path) as f:
        for line in f.readlines():
          if line[:6] != "<body>":
            return line
  except:
    print(path)
    return ""

  return ""


def read_files(dir_path):
  path_list = os.listdir(dir_path)
  content = ""

  for path in path_list:
    content += read_file(dir_path + "/" + path)

  return content

In [3]:
datalake_path = "dataset"

In [4]:
text_data = read_files(datalake_path)

dataset/https__github.com_doocs_advanced-java_blob_main_main.js
dataset/https__github.com_iluwatar_java-design-patterns_blob_master_game-loop_src_main_java_com_iluwatar_gameloop_FixedStepGameLoop.java


Now we split the plain text into train and test data, and store it on drive

In [5]:
train_text = text_data[:int(0.8*len(text_data))]
test_text = text_data[int(0.8*len(text_data)):]

In [6]:
path = ""

with open(path + "train_text.txt", "w") as f:
  f.write(train_text)

with open(path + "test_text.txt", "w") as f:
  f.write(test_text)

# Retraining GPT-2 (Fine tuning)

In this section, we will fine tune GPT-2 using python scripts, all of them obtained via **GitHub Scrapper for RePylot**

In [7]:
from transformers import TextDataset, DataCollatorForLanguageModeling
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from transformers import Trainer, TrainingArguments

import torch

In [8]:
def load_dataset(file_path, tokenizer, block_size = 128):
    dataset = TextDataset(
        tokenizer = tokenizer,
        file_path = file_path,
        block_size = block_size,
    )
    return dataset


def load_data_collator(tokenizer, mlm = False):
    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=mlm,
    )
    return data_collator

In [9]:
def train(train_file_path,model_name,
          output_dir,
          overwrite_output_dir,
          per_device_train_batch_size,
          num_train_epochs,
          save_steps):
  tokenizer = GPT2Tokenizer.from_pretrained(model_name)
  tokenizer.save_pretrained(output_dir)

  train_dataset = load_dataset(train_file_path, tokenizer)
  data_collator = load_data_collator(tokenizer)

  model = GPT2LMHeadModel.from_pretrained(model_name)
  
  device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
  model.to(device)
  model.save_pretrained(output_dir)

  training_args = TrainingArguments(
          output_dir=output_dir,
          overwrite_output_dir=overwrite_output_dir,
          per_device_train_batch_size=per_device_train_batch_size,
          num_train_epochs=num_train_epochs,
          save_steps=save_steps,
      )
  
  print(model.device)
  trainer = Trainer(
          model=model,
          args=training_args,
          data_collator=data_collator,
          train_dataset=train_dataset,
  )

  trainer.train()
  trainer.save_model()

In [10]:
train_file_path = "train_text.txt"
model_name = 'gpt2'

output_dir = 'models/custom_gpt2'
overwrite_output_dir = True
per_device_train_batch_size = 8
num_train_epochs = 5
save_steps = 0

In [11]:
train(
    train_file_path=train_file_path,
    model_name=model_name,
    output_dir=output_dir,
    overwrite_output_dir=overwrite_output_dir,
    per_device_train_batch_size=per_device_train_batch_size,
    num_train_epochs=num_train_epochs,
    save_steps=save_steps
)

C:\Users\carde\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.9_qbz5n2kfra8p0\LocalCache\local-packages\Python39\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
C:\Users\carde\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.9_qbz5n2kfra8p0\LocalCache\local-packages\Python39\site-packages\transformers\data\datasets\language_modeling.py:53: FutureWarning: This dataset will be removed from the library soon, preprocessing should be handled with the 🤗 Datasets library. You can have a look at this example script for pointers: https://github.com/huggingface/transformers/blob/main/examples/pytorch/language-modeling/run_mlm.py
  warnings.warn(


cuda:0


 19%|█▊        | 500/2695 [08:29<43:06,  1.18s/it]

{'loss': 1.3206, 'grad_norm': 2.9556143283843994, 'learning_rate': 4.072356215213358e-05, 'epoch': 0.93}


 37%|███▋      | 1000/2695 [23:05<32:16,  1.14s/it] 

...

{'loss': 0.7798, 'grad_norm': 4.038917541503906, 'learning_rate': 1.2894248608534323e-05, 'epoch': 3.71}


 93%|█████████▎| 2500/2695 [1:03:11<05:43,  1.76s/it]

{'loss': 0.7252, 'grad_norm': 2.581815242767334, 'learning_rate': 3.6178107606679037e-06, 'epoch': 4.64}


100%|██████████| 2695/2695 [1:08:59<00:00,  1.54s/it]


{'train_runtime': 4139.586, 'train_samples_per_second': 5.201, 'train_steps_per_second': 0.651, 'train_loss': 0.91506334067718, 'epoch': 5.0}


# Model Evaluation

We can now proceed testing the model we have just trained

In [12]:
def load_model(model_path):
    model = GPT2LMHeadModel.from_pretrained(model_path)
    return model


def load_tokenizer(tokenizer_path):
    tokenizer = GPT2Tokenizer.from_pretrained(tokenizer_path)
    return tokenizer


def generate_text(model_path, sequence, extra_length):
    model = load_model(model_path).to(torch.device("cuda"))
    model.requires_grad_(False)
    tokenizer = load_tokenizer(model_path)

    ids = tokenizer.encode(f'{sequence}', return_tensors='pt').to(torch.device("cuda"))
    final_outputs = model.generate(
        ids,
        do_sample=True,
        max_length=len(ids) + extra_length,
        pad_token_id=model.config.eos_token_id,
        top_k=50,
        top_p=0.95,
    )

    print(tokenizer.decode(final_outputs[0], skip_special_tokens=True))

Feel free to modify `sequence` variable in order to test the model yourself

In [44]:
sequence = "public class"
generate_text(output_dir, sequence, extra_length=20)

public class CityTests { @Test void defaultPropertyPlaceholders() { this.contextRunner.run


In [17]:
sequence = "public static void"
generate_text(output_dir, sequence, extra_length=20)

public static void main(String[] args) { String description = "The user name is: %s


In [20]:
sequence = "@Override"
generate_text(output_dir, sequence, extra_length=20)

@Override public String toString() { return this.name; } }
 /* * Copyright 2012-


In [30]:
sequence = "for (String link :"
generate_text(output_dir, sequence, extra_length=20)

for (String link : null) throws SQLException { LOGGER.info(link.get


If we now compare the obtained result with the original GPT2 model output, we can appreciate how a better response is achieved by our fine tuned transformer. Moreover, the code generated by RePylot is indeed a Java code. Meanwhile, original GPT2 generated code in other language, which possibly doesn't even exist

In [31]:
from transformers import pipeline, set_seed
generator = pipeline('text-generation', model='gpt2', device=torch.device("cuda"))

C:\Users\carde\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.9_qbz5n2kfra8p0\LocalCache\local-packages\Python39\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [41]:
sequence = "public class"
generator(sequence, max_length=30, num_return_sequences=1)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


[{'generated_text': 'public class Bool is a constructor function that provides an argument that you can use to define new value types for your classes:\n\nclass Bool'}]

Other examples are the following

In [45]:
sequence = "public class"

print("RePylot generation:")
generate_text(output_dir, sequence, extra_length=20)

print("\nGPT-2 generation:")
generator(sequence, max_length=30, num_return_sequences=1)

RePylot generation:


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


public class SecurityReactiveCredentialProvider implements CassandraReactiveCredentialProvider<securitycred

GPT-2 generation:


[{'generated_text': 'public class Person(object sender, EventArgs e) { if (e.name.getAttribute("id")== \'person\' || e.'}]

As can be seen, the fine tuned model is able to generate code that is more similar to Java. The original GPT2 model, on the other hand, generates code that is not even a valid syntax.

In [46]:
sequence = "import"

print("RePylot generation:")
generate_text(output_dir, sequence, extra_length=20)

print("\nGPT-2 generation:")
generator(sequence, max_length=30, num_return_sequences=1)

RePylot generation:


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


import org.springframework.boot.actuate.endpoint.annotation.Endpoint; import

GPT-2 generation:


[{'generated_text': 'import the default settings.\n\nFor example, to allow a user to choose which mode to use, add, and exit the default settings.\n'}]

Note that these results have been obtained fine tuning GPT-2 in only 5 epochs. Due to the impresive results, we can expect even better results by increasing the number of epochs. Thus, we resume the training process

In [47]:
train_file_path = "train_text.txt"
model_name = 'models/custom_gpt2'

output_dir = 'models/custom_gpt2_10'
overwrite_output_dir = True
per_device_train_batch_size = 8
num_train_epochs = 10
save_steps = 0

In [48]:
train(
    train_file_path=train_file_path,
    model_name=model_name,
    output_dir=output_dir,
    overwrite_output_dir=overwrite_output_dir,
    per_device_train_batch_size=per_device_train_batch_size,
    num_train_epochs=num_train_epochs,
    save_steps=save_steps
)

C:\Users\carde\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.9_qbz5n2kfra8p0\LocalCache\local-packages\Python39\site-packages\transformers\data\datasets\language_modeling.py:53: FutureWarning: This dataset will be removed from the library soon, preprocessing should be handled with the 🤗 Datasets library. You can have a look at this example script for pointers: https://github.com/huggingface/transformers/blob/main/examples/pytorch/language-modeling/run_mlm.py
  warnings.warn(


cuda:0


  0%|          | 25/5390 [01:00<3:47:41,  2.55s/it]

KeyboardInterrupt: 

If we now evaluate the model again, we can appreciate how the model has improved its performance

In [25]:
sequence = "for m"
generate_text('models/custom_gpt2_10', sequence, extra_length=20)

for m in t.parameters[m].items(): assert len(parameters) &gt;


In [43]:
sequence = "from matplotlib import"
generate_text(output_dir, sequence, extra_length=20)

from matplotlib import pyplot as plt from sklearn.metrics import accuracy def accuracy_


In [145]:
sequence = "for key in"
generate_text('models/custom_gpt2', sequence, extra_length=20)

for key in ciphertext) if not key.contains("C") and not decrypt_key:


In [131]:
sequence = "def inverse_sort(list, number):"
generate_text('models/custom_gpt2', sequence, extra_length=50)

def inverse_sort(list, number): return [list[float] for _ in range(number)] def inverse_sort(list, number): return []
 """ https://en.wikipedia.org/wiki/List_of_prime_


In [132]:
sequence = "def inverse_sort(list, number):"
generate_text('models/custom_gpt2_10', sequence, extra_length=50)

def inverse_sort(list, number): if number &lt;= 0: raise ValueError("numbers cannot be negative") return inverse_sort(list, number) def is_safe(self): return len(self.values) ==


In [148]:
sequence = "if (a =="
generate_text('models/custom_gpt2_10', sequence, extra_length=20)

if (a == a[0]) # check if the queue is empty assert len(a)!=


In [149]:
sequence = "if (a =="
generate_text(output_dir, sequence, extra_length=20)

if (a == b) and isinstance(i, (int, list)): if arr.


Now, we try asking the model to generate Java code, where failure is expected

In [ ]:
sequence = "for m"
generate_text('models/custom_gpt2_10', sequence, extra_length=20)

for m in t.parameters[m].items(): assert len(parameters) &gt;


Now we ask the model to generate Java code, where failure is expected

In [29]:
sequence = "public class Main {"
generate_text('models/custom_gpt2_10', sequence, extra_length=20)

public class Main { output: str input: str output_filename: str output_key: str }


In [30]:
sequence = "public static void"
generate_text('models/custom_gpt2_10', sequence, extra_length=20)

public static void __bool__( self, binary: bool = False,... ai_profile:


In [31]:
sequence = "ArrayList<String> list = new"
generate_text('models/custom_gpt2_10', sequence, extra_length=20)

ArrayList<String> list = new HashMap(5) for idx, item in enumerate
